## Surrogate vs. Thermochemical library
### **Computational speedup**
Import libraries

In [1]:
import os
import sys
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from tqdm import tqdm

In [2]:
sys.path.append(os.path.abspath('./../functions'))

import plotting as plt

%load_ext kernel_manager

Define inputs

In [3]:
nb_cells = [100, 1000, 5000, 10000, 20000, 50000, 100000]
reps = 100
# Input space
ic = {
    'rho_O': 6.0e-3,
    'rho_O2': 5.9e-2,
    'Th': 3200,
    'Tv': 1000
}
dt = 5.e-8
# Working case
path_to_case = "./../../"

In [4]:
skip_flags = {
    'model': True
}

Set paths

In [5]:
path_to_dataset = os.path.abspath(path_to_case + f'/tools/datasets/comp_time/')
os.makedirs(path_to_dataset, exist_ok=True)
path_to_saving = os.path.abspath("./../output/comp_time/")
os.makedirs(path_to_saving, exist_ok=True)

### Thermochemical library computational time
Read times

In [6]:
def read_times(path, nb_cells):
    times = {'nb_cells': nb_cells, 'mean': [], 'std': []}
    for n in nb_cells:
        t = np.loadtxt(path + f"/cpu_time.{n}.dat")
        times['mean'].append(np.mean(t))
        times['std'].append(np.std(t))
    return times

In [7]:
path = path_to_case + '/pde/thermolib/comp_time/data/'
filename = path_to_dataset + f'/thermolib.csv'
if os.path.exists(filename):
    df = pd.read_csv(filename)
else:
    times = read_times(path, nb_cells)
    df = pd.DataFrame.from_dict(times)
    df.to_csv(filename, index=False)
thermolib_time = df

### Surrogate computational time
Global variables

In [8]:
%%skip $skip_flags['model']

# Set number of cores
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
# Load models
model_path = path_to_case + '/pde/surrogate/model/SavedModel'
model = tf.keras.models.load_model(filepath=model_path)
# Compose inputs
inputs = np.array(list(ic.values())+[dt])

Evaluate inference time

In [9]:
def eval_times(model, inputs, nb_cells, reps):
    times = {'nb_cells': nb_cells, 'mean': [], 'std': []}
    for n in tqdm(nb_cells):
        t = []
        i = np.tile(inputs, (n, 1))
        for _ in range(reps):
            start = time.time()
            y_pred = model(i)
            end = time.time()
            t.append(end-start)
        times['mean'].append(np.mean(t))
        times['std'].append(np.std(t))
    return times

In [10]:
surrogate_time = {}
for device in ('GPU', 'CPU'):
    filename = path_to_dataset + f'/surrogate.{device.lower()}.csv'
    if os.path.exists(filename):
        df = pd.read_csv(filename)
    else:
        with tf.device(f'/{device}:0'):
            times = eval_times(model, inputs, nb_cells, reps)
        df = pd.DataFrame.from_dict(times)
        df.to_csv(filename, index=False)
    surrogate_time[device] = df

### Speedup

In [11]:
ref_time = thermolib_time['mean'].to_numpy()[1:]
data = {k: ref_time / t['mean'].to_numpy()[1:] for k, t in surrogate_time.items()}

In [12]:
plt.speedup(
    nb_cells[1:],
    data,
    figname='speedup',
    path=path_to_saving
)